## 1. 스트리밍 수 예측 모델

### 데이터 전처리

In [98]:
import pandas as pd

data = pd.read_csv("/content/Spotify_Youtube.csv")
data.head()

,Unnamed: 0,Artist,Url_spotify,Track,Album,Album_type,Uri,Danceability,Energy,Key,...,Url_youtube,Title,Channel,Views,Likes,Comments,Description,Licensed,official_video,Stream
0,0,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Feel Good Inc.,Demon Days,album,spotify:track:0d28khcov6AiegSCpG5TuT,0.818,0.705,6.0,...,https://www.youtube.com/watch?v=HyHNuVaZJ-k,Gorillaz - Feel Good Inc. (Official Video),Gorillaz,693555221.0,6220896.0,169907.0,Official HD Video for Gorillaz' fantastic trac...,True,True,1.040235e+09
1,1,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Rhinestone Eyes,Plastic Beach,album,spotify:track:1foMv2HQwfQ2vntFf9HFeG,0.676,0.703,8.0,...,https://www.youtube.com/watch?v=yYDmaexVHic,Gorillaz - Rhinestone Eyes [Storyboard Film] (...,Gorillaz,72011645.0,1079128.0,31003.0,The official video for Gorillaz - Rhinestone E...,True,True,3.100837e+08
2,2,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,New Gold (feat. Tame Impala and Bootie Brown),New Gold (feat. Tame Impala and Bootie Brown),single,spotify:track:64dLd6rVqDLtkXFYrEUHIU,0.695,0.923,1.0,...,https://www.youtube.com/watch?v=qJa-VFwPpYA,Gorillaz - New Gold ft. Tame Impala & Bootie B...,Gorillaz,8435055.0,282142.0,7399.0,Gorillaz - New Gold ft. Tame Impala & Bootie B...,True,True,6.306347e+07
3,3,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,On Melancholy Hill,Plastic Beach,album,spotify:track:0q6LuUqGLUiCPP1cbdwFs3,0.689,0.739,2.0,...,https://www.youtube.com/watch?v=04mfKJWDSzI,Gorillaz - On Melancholy Hill (Official Video),Gorillaz,211754952.0,1788577.0,55229.0,Follow Gorillaz online:\nhttp://gorillaz.com \...,True,True,4.346636e+08
4,4,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Clint Eastwood,Gorillaz,album,spotify:track:7yMiX7n9SBvadzox8T5jzT,0.663,0.694,10.0,...,https://www.youtube.com/watch?v=1V_xRb0x9aw,Gorillaz - Clint Eastwood (Official Video),Gorillaz,618480958.0,6197318.0,155930.0,The official music video for Gorillaz - Clint ...,True,True,6.172597e+08


In [99]:
#필요 없는 컬럼 삭제
# 삭제할 컬럼
cols_to_drop = ['Unnamed: 0', 'Url_spotify', 'Uri', 'Url_youtube', 'Description', 'Licensed','Title','Channel','official_video']

# 컬럼을 삭제한 데이터를 music에 저장
music = data.drop(columns=cols_to_drop)

music.head()

,Artist,Track,Album,Album_type,Danceability,Energy,Key,Loudness,Speechiness,Acousticness,Instrumentalness,Liveness,Valence,Tempo,Duration_ms,Views,Likes,Comments,Stream
0,Gorillaz,Feel Good Inc.,Demon Days,album,0.818,0.705,6.0,-6.679,0.1770,0.008360,0.002330,0.6130,0.772,138.559,222640.0,693555221.0,6220896.0,169907.0,1.040235e+09
1,Gorillaz,Rhinestone Eyes,Plastic Beach,album,0.676,0.703,8.0,-5.815,0.0302,0.086900,0.000687,0.0463,0.852,92.761,200173.0,72011645.0,1079128.0,31003.0,3.100837e+08
2,Gorillaz,New Gold (feat. Tame Impala and Bootie Brown),New Gold (feat. Tame Impala and Bootie Brown),single,0.695,0.923,1.0,-3.930,0.0522,0.042500,0.046900,0.1160,0.551,108.014,215150.0,8435055.0,282142.0,7399.0,6.306347e+07
3,Gorillaz,On Melancholy Hill,Plastic Beach,album,0.689,0.739,2.0,-5.810,0.0260,0.000015,0.509000,0.0640,0.578,120.423,233867.0,211754952.0,1788577.0,55229.0,4.346636e+08
4,Gorillaz,Clint Eastwood,Gorillaz,album,0.663,0.694,10.0,-8.627,0.1710,0.025300,0.000000,0.0698,0.525,167.953,340920.0,618480958.0,6197318.0,155930.0,6.172597e+08


In [100]:
music = music.dropna(subset=['Album_type', 'Danceability'])

In [101]:
music = music.dropna(subset=['Stream'])

In [102]:
# 결측 처리
for col in ['Views', 'Likes', 'Comments']:
    music[col] = music[col].fillna(0)

In [103]:
music.isnull().sum()

,0
Artist,0
Track,0
Album,0
Album_type,0
Danceability,0
Energy,0
Key,0
Loudness,0
Speechiness,0
Acousticness,0


In [104]:
music.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20140 entries, 0 to 20717
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Artist            20140 non-null  object 
 1   Track             20140 non-null  object 
 2   Album             20140 non-null  object 
 3   Album_type        20140 non-null  object 
 4   Danceability      20140 non-null  float64
 5   Energy            20140 non-null  float64
 6   Key               20140 non-null  float64
 7   Loudness          20140 non-null  float64
 8   Speechiness       20140 non-null  float64
 9   Acousticness      20140 non-null  float64
 10  Instrumentalness  20140 non-null  float64
 11  Liveness          20140 non-null  float64
 12  Valence           20140 non-null  float64
 13  Tempo             20140 non-null  float64
 14  Duration_ms       20140 non-null  float64
 15  Views             20140 non-null  float64
 16  Likes             20140 non-null  float64
 17

In [75]:
# music.to_csv('music.csv', index=False)

### EDA

In [105]:
#범주형 분석(앨범 타입)
music.groupby('Album_type')['Stream'].mean()

,Stream
Album_type,
album,1.499812e+08
compilation,8.400785e+07
single,1.016708e+08


In [106]:
music.groupby('Artist')['Stream'].mean().sort_values(ascending=False)

,Stream
Artist,
Post Malone,1.525126e+09
Ed Sheeran,1.439488e+09
Dua Lipa,1.340808e+09
XXXTENTACION,1.322435e+09
The Weeknd,1.303197e+09
...,...
Traditional,1.949461e+05
Teufelskicker,1.115469e+05
Sherlock Holmes,1.046010e+05


In [107]:
import numpy as np

for col in ['Stream', 'Views', 'Likes', 'Comments']:
    music[col] = np.log1p(music[col])

In [108]:
music.head(3)

,Artist,Track,Album,Album_type,Danceability,Energy,Key,Loudness,Speechiness,Acousticness,Instrumentalness,Liveness,Valence,Tempo,Duration_ms,Views,Likes,Comments,Stream
0,Gorillaz,Feel Good Inc.,Demon Days,album,0.818,0.705,6.0,-6.679,0.1770,0.00836,0.002330,0.6130,0.772,138.559,222640.0,20.357341,15.643425,12.043012,20.762712
1,Gorillaz,Rhinestone Eyes,Plastic Beach,album,0.676,0.703,8.0,-5.815,0.0302,0.08690,0.000687,0.0463,0.852,92.761,200173.0,18.092338,13.891665,10.341872,19.552353
2,Gorillaz,New Gold (feat. Tame Impala and Bootie Brown),New Gold (feat. Tame Impala and Bootie Brown),single,0.695,0.923,1.0,-3.930,0.0522,0.04250,0.046900,0.1160,0.551,108.014,215150.0,15.947907,12.550169,8.909235,17.959652


###데이터 분할

In [83]:
from sklearn.model_selection import train_test_split

x = music.drop(columns=['Stream'])
y = music['Stream']

x_train, x_test, y_train, y_test = train_test_split(x, y)

###모델 생성(catboost)

In [84]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.4 MB/s eta 0:00:00


In [85]:
import numpy as np
from catboost import Pool, cv

# 2️⃣ 범주형 컬럼 지정
cat_features = ['Artist', 'Track', 'Album', 'Album_type']

# 3️⃣ CatBoost 전용 데이터셋 생성
train_pool = Pool(
    data=x,
    label=y,
    cat_features=cat_features
)

# 4️⃣ 파라미터 설정
params = {
    'loss_function': 'RMSE',   # 회귀 문제 → RMSE 사용
    'iterations': 1000,        # 최대 학습 횟수
    'learning_rate': 0.05,     # 학습 속도
    'depth': 6,                # 트리 깊이
    'random_seed': 42,
    'verbose': False
}

# 5️⃣ 교차검증 수행
cv_result = cv(
    pool=train_pool,
    params=params,
    fold_count=5,                  # 5-fold CV
    early_stopping_rounds=50,      # 성능 안 좋아지면 조기 종료
    verbose=False
)

# 6️⃣ 결과 확인
best_rmse = cv_result['test-RMSE-mean'].min()
best_iter = cv_result['test-RMSE-mean'].idxmin()

print(f"최적 iteration: {best_iter}")
print(f"최소 RMSE: {best_rmse}")

Training on fold [0/5]

bestTest = 0.9426625982
bestIteration = 999

Training on fold [1/5]

bestTest = 0.9527659264
bestIteration = 999

Training on fold [2/5]

bestTest = 0.9561924565
bestIteration = 978

Training on fold [3/5]

bestTest = 0.9228723436
bestIteration = 993

Training on fold [4/5]

bestTest = 0.9659457374
bestIteration = 999

최적 iteration: 999
최소 RMSE: 0.948173293915781


In [86]:
#모델 평가
cv_result['test-RMSE-mean'].min()


0.948173293915781

In [87]:
#최적의 iter로 학습
from catboost import CatBoostRegressor

model = CatBoostRegressor(
    iterations=best_iter,
    learning_rate=0.05,
    depth=6,
    verbose=0
)

model.fit(x, y, cat_features=cat_features)

y_pred = model.predict(x_test)

In [88]:
#RMSE
from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE:", rmse)

RMSE: 0.6917522435962625


In [89]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)
print("R2:", r2)

R2: 0.8262518465175879


In [91]:
#feature importances
import pandas as pd

importance = model.get_feature_importance()
feature_names = x.columns

pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values(by='importance', ascending=False)

,feature,importance
0,Artist,19.347615
15,Views,18.048801
16,Likes,17.473743
2,Album,6.603769
17,Comments,5.806520
3,Album_type,5.711465
8,Speechiness,5.514216
14,Duration_ms,4.491565
9,Acousticness,2.646800
7,Loudness,2.220125


###모델 최적화

In [92]:
#그리드 서치
from catboost import CatBoostRegressor
from sklearn.model_selection import GridSearchCV

cat_features = ['Artist','Track','Album','Album_type']

model = CatBoostRegressor(
    verbose=0,
    random_seed=42
)

param_grid = {
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3, 5],
    'iterations': [300, 500, 800]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid.fit(x_train, y_train, cat_features=cat_features)

print("최적 파라미터:", grid.best_params_)
print("최적 RMSE:", -grid.best_score_)

최적 파라미터: {'depth': 8, 'iterations': 800, 'l2_leaf_reg': 3, 'learning_rate': 0.05}
최적 RMSE: 0.9999653039010815


In [93]:
#모델학습
best_model = grid.best_estimator_

best_model.fit(x_train, y_train, cat_features=cat_features)

CatBoostRegressor(depth=8, iterations=800, l2_leaf_reg=3, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=0)

In [94]:
#평가
from sklearn.metrics import mean_squared_error
import numpy as np

y_pred = best_model.predict(x_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Test RMSE:", rmse)

r2 = r2_score(y_test, y_pred)
print("R2:", r2)

Test RMSE: 0.9665865731375234
R2: 0.6607650721415561


그리드 서피로 하이퍼파라미터를 최적화한 성능보다 기존 모델의 성능이 더 좋기 때문에 기존 모델 채택

스트리밍 수에 가장 영향을 많이 주는 요소는 Artist이다  
또한 유튜브 뮤직 비디오 관련 데이터와 스포티파이 스트리밍수가 양의 상관관계를 갖는 것을 알수 있다

하지만 위 데이터는 컬럼이 20개 정도 밖에 없어서 정확이 스트리밍 수를 맞추기에는 한계가 있다  
더 많은 요인과 요소가 있어야 더 정확한 예측이 가능하다

##음악 추천 모델

In [109]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [112]:
music = pd.read_csv('/content/music.csv')
my_data = pd.read_csv('/content/my_data.csv')

#가중치를 줄 것임 -> 호감도와 재생 횟수
#가중치 = 호감도 * 재생 횟수
#재생 횟수의 차이가 크기 때문에 스케일링 진행
#재생 횟수가 집계 되지 않은 음악도 있기 때문에 1을 더해서 가중치가 0이되지 않도록 할 것임
my_data['Count_scaled'] = np.log1p(my_data['Count'])
my_data['Weight'] = my_data['Score'] * (1 + my_data['Count_scaled'])

my_data.head(3)

,Artist,Track,Album,Album_type,Danceability,Energy,Key,Loudness,Speechiness,Acousticness,...,Duration_ms,Views,Likes,Comments,official_video,Stream,Score,Count,Count_scaled,Weight
0,Bruno Mars,When I Was Your Man,Unorthodox Jukebox,album,0.612,0.280,0.0,-8.648,0.0434,0.9320,...,213827.0,1.237507e+09,6771190.0,193121.0,True,1.439567e+09,5,0,0.000000,5.000000
1,Miley Cyrus,Flowers,Flowers,single,0.707,0.681,0.0,-4.325,0.0668,0.0632,...,200455.0,1.545193e+08,4625162.0,117998.0,True,2.838849e+08,5,0,0.000000,5.000000
2,Maroon 5,Memories,JORDI (Deluxe),album,0.775,0.327,11.0,-7.241,0.0557,0.8410,...,189486.0,9.018294e+08,9050321.0,265339.0,True,1.651773e+09,5,278,5.631212,33.156059


In [115]:
#결측치 처리
music = music.drop(columns=['official_video'])
music = music.dropna(subset=['Album_type', 'Danceability'])

for col in ['Views', 'Likes', 'Comments','Stream']:
    music[col] = music[col].fillna(0)

music.isnull().sum()

,0
Artist,0
Track,0
Album,0
Album_type,0
Danceability,0
Energy,0
Key,0
Loudness,0
Speechiness,0
Acousticness,0


###코사인 유사도

In [116]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

#특성들을 스케일링 해줍니다!

#콘텐츠 기반 추천을 하기 위해서 메타 데이터, 범주형, 인기 지표, 사용자 지표를 제외한 특성을 고르겠습니다.
features = [
'Danceability','Energy','Loudness','Speechiness',
'Acousticness','Instrumentalness','Liveness',
'Valence','Tempo','Duration_ms'
]

#music 데이터를 스케일링 해줍니다
scaler = StandardScaler()

music_scaled = scaler.fit_transform(music[features])

#데이터 프레임으로 변환
music_scaled = pd.DataFrame(
    music_scaled,
    columns=features,
    index=music.index
)

#my_data도 스케일링
my_scaled = scaler.transform(my_data[features])

my_scaled = pd.DataFrame(
    my_scaled,
    columns=features,
    index=my_data.index
)

#사용자 취향 벡터 생성
X = my_scaled.values
w = my_data['Weight'].values.reshape(-1,1)

user_profile = (X * w).sum(axis=0) / w.sum()

#전체 음악과 코사인 유사도 계산
similarity = cosine_similarity(
    [user_profile],
    music_scaled
)

#추천 결과 생성
music['similarity'] = similarity[0]

recommend = music.sort_values(
    'similarity',
    ascending=False
)

#이미 들었던 음악은 제거하고
recommend = recommend[
    ~recommend['Track'].isin(my_data['Track'])
]

#추천하는 음악 10개 출력
recommend.head(10)[['Track','Artist','similarity']]

,Track,Artist,similarity
15805,Bad At Love,Halsey,0.922660
13416,Give Your Heart a Break,Demi Lovato,0.921801
18148,Little Bit of Love,Tom Grennan,0.897700
6527,All Stars,Martin Solveig,0.894325
13852,Bones,Imagine Dragons,0.890686
16681,Follow Me,Sam Feldt,0.889972
14656,Follow Me,Rita Ora,0.889972
18062,Always Be There,Jonas Blue,0.884721
16684,Call On Me (feat. Georgia Ku),Sam Feldt,0.882170
17105,Just A Stranger (feat. Steve Lacy),Kali Uchis,0.882027


###호감도 예측 회귀 모델

In [117]:
#라이브러리 임포트
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

In [123]:
#feature 선택과 처리하지 못한 결측치 전처리
features = [
'Danceability','Energy','Loudness','Speechiness',
'Acousticness','Instrumentalness','Liveness',
'Valence','Tempo'
]

music[features] = music[features].fillna(music[features].mean())
my_data[features] = my_data[features].fillna(music[features].mean())

#feature 스케일링
scaler = StandardScaler()

music_scaled = scaler.fit_transform(music[features])
my_scaled = scaler.transform(my_data[features])

In [124]:
#학습 할 데이터 셋을 만들어 주겠습니다
#스케일링한 음악 특성이 독립변수, 호감도인 Score가 종속변수가 될 것입니다
X = my_scaled
y = my_data['Score']

#재생 횟수를 가중치로 주겠습니다
sample_weight = 1 + np.log1p(my_data['Count'])

In [125]:
#train test split해줍니다! 가중치도 함께!
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(X,y,sample_weight,test_size=0.2)

In [126]:
#회귀 모델 만들기
#가장 기본적인 모델로 RandomForest를 사용했습니다
model = RandomForestRegressor(n_estimators=200)

model.fit(X_train,y_train,sample_weight=w_train)

#예측
y_pred = model.predict(X_test)

#이제 여러 지표로 평가해보겠습니다!

#mae
mae = mean_absolute_error(y_test, y_pred)

#rmse
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

#r^2
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 1.078076797385621
RMSE: 1.2055667403512658
R2: -0.018940814369946102


###다른 모델 사용해보기! catboost

In [147]:
my = pd.read_csv('/content/my_data.csv')

In [133]:
#로그변환
# 'Stream', 'Views', 'Likes', 'Comments' 컬럼은 music DataFrame에 적용
for col in ['Stream', 'Views', 'Likes', 'Comments']:
    [col] = np.log1p(music[col])

# 'Count' 컬럼은 my DataFrame에 적용
my['Count'] = np.log1p(my['Count'])

In [148]:
#로그변환
for col in ['Stream', 'Views', 'Likes', 'Comments']:
    my[col] = np.log1p(my[col])

# 'Count' 컬럼은 my DataFrame에 적용
my['Count'] = np.log1p(my['Count'])

In [149]:
from sklearn.model_selection import train_test_split

x = my.drop(columns=['Score'])
y = my['Score']

x_train, x_test, y_train, y_test = train_test_split(x, y)

In [150]:
import numpy as np
from catboost import Pool, cv

# 2️⃣ 범주형 컬럼 지정
cat_features = ['Artist', 'Track', 'Album', 'Album_type','official_video']

# 3️⃣ CatBoost 전용 데이터셋 생성
train_pool = Pool(
    data=x,
    label=y,
    cat_features=cat_features
)

# 4️⃣ 파라미터 설정
params = {
    'loss_function': 'RMSE',   # 회귀 문제 → RMSE 사용
    'iterations': 1000,        # 최대 학습 횟수
    'learning_rate': 0.05,     # 학습 속도
    'depth': 6,                # 트리 깊이
    'random_seed': 42,
    'verbose': False
}

# 5️⃣ 교차검증 수행
cv_result = cv(
    pool=train_pool,
    params=params,
    fold_count=5,                  # 5-fold CV
    early_stopping_rounds=50,      # 성능 안 좋아지면 조기 종료
    verbose=False
)

# 6️⃣ 결과 확인
best_rmse = cv_result['test-RMSE-mean'].min()
best_iter = cv_result['test-RMSE-mean'].idxmin()

print(f"최적 iteration: {best_iter}")
print(f"최소 RMSE: {best_rmse}")

Training on fold [0/5]

bestTest = 1.124689528
bestIteration = 209

Training on fold [1/5]

bestTest = 1.037639007
bestIteration = 216

Training on fold [2/5]

bestTest = 1.081803833
bestIteration = 146

Training on fold [3/5]

bestTest = 1.086895872
bestIteration = 378

Training on fold [4/5]

bestTest = 0.9792528622
bestIteration = 475

최적 iteration: 475
최소 RMSE: 1.0681324854393193


In [151]:
#모델 평가
cv_result['test-RMSE-mean'].min()

1.0681324854393193

In [152]:
#최적의 iter로 학습
from catboost import CatBoostRegressor

model = CatBoostRegressor(
    iterations=best_iter,
    learning_rate=0.05,
    depth=6,
    verbose=0
)

model.fit(x, y, cat_features=cat_features)

y_pred = model.predict(x_test)

In [153]:
#RMSE
from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE:", rmse)

RMSE: 0.30447247473663225


In [154]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)
print("R2:", r2)

R2: 0.9523212598788633


In [155]:
#feature importances
import pandas as pd

importance = model.get_feature_importance()
feature_names = x.columns

pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values(by='importance', ascending=False)

,feature,importance
0,Artist,15.420172
20,Count,15.138280
8,Speechiness,6.871814
7,Loudness,5.259161
9,Acousticness,5.130042
15,Views,5.108849
6,Key,4.811121
5,Energy,4.516543
12,Valence,4.505916
4,Danceability,4.241575


이 모델을 호감도가 없는 music데이터셋에 적용해서 호감도를 예측하고
높은 호감도를 가진 음악 10개를 추천해보겠습니다.

###음악 추천

In [157]:
#music 데이터셋 전처리
music = pd.read_csv('/content/music.csv')

#필요 없는 컬럼 삭제
music = music.dropna(subset=['official_video'])

#로그변환
for col in ['Stream', 'Views', 'Likes', 'Comments']:
    music[col] = np.log1p(music[col])

# 'Count' 컬럼만들기
music['Count'] = 0

music.head(3)

,Artist,Track,Album,Album_type,Danceability,Energy,Key,Loudness,Speechiness,Acousticness,...,Liveness,Valence,Tempo,Duration_ms,Views,Likes,Comments,official_video,Stream,Count
0,Gorillaz,Feel Good Inc.,Demon Days,album,0.818,0.705,6.0,-6.679,0.1770,0.00836,...,0.6130,0.772,138.559,222640.0,20.357341,15.643425,12.043012,True,20.762712,0
1,Gorillaz,Rhinestone Eyes,Plastic Beach,album,0.676,0.703,8.0,-5.815,0.0302,0.08690,...,0.0463,0.852,92.761,200173.0,18.092338,13.891665,10.341872,True,19.552353,0
2,Gorillaz,New Gold (feat. Tame Impala and Bootie Brown),New Gold (feat. Tame Impala and Bootie Brown),single,0.695,0.923,1.0,-3.930,0.0522,0.04250,...,0.1160,0.551,108.014,215150.0,15.947907,12.550169,8.909235,True,17.959652,0


In [158]:
#호감도 예측
y_pred = model.predict(music)

In [159]:
music['predicted_score'] = y_pred

In [161]:
top10 = music.sort_values(
    by='predicted_score',
    ascending=False
).head(10)

top10[['Artist', 'Track', 'predicted_score']]

,Artist,Track,predicted_score
17674,Post Malone,I Fall Apart,3.993363
10959,Bob Seger,Mainstreet,3.856559
19226,Jordan Davis,What My World Spins Around,3.854504
16488,Lizzo,Special,3.823758
244,Sia,Del Mar,3.799905
7188,D.O.D,Sleepless,3.798271
15179,DJ Snake,U Are My High,3.782594
2605,The Jackson 5,I Saw Mommy Kissing Santa Claus,3.774930
4565,Boston,Rock & Roll Band,3.749584
12335,Ed Sheeran,Celestial,3.749350


위 모델보다 더 선호하는 가수를 중심으로 추천이 되었다!
하지만 음악 추천 모델을 평가하기가 어렵다
사용자의 취향에 맞아야하는데 취향을 수치화하기 어렵기 때문!
따라서 이 모델을 평가할 수 있는 방법에 대해 더 생각해야할 것 같다

Pilot 프로젝트를 하면서 아쉬웠던 점!
일단 데이터셋이 작다. 내 음악 청취 데이터셋은 200개밖에 되지 않는다 따라서 이후에는 스포티파이 API를 가져와서 음악 청취 데이터를 크게 키워 진행해보는 것도 좋을 것 같다

그리고 평가 지표가 아쉽다.
앞에서 말한 것 처럼 음악을 추천해주고 이게 사용자가 좋아하는지 확인하는 기준이 매우 모호하다. 따라서 이에 대한 기준도 다시 생각해야할 것 같다